# Evaluación del modelo v0 de LLIKA: 2022 y 2024

Este notebook rehace, **sin reentrenar nada**, la evaluación que resume la Model Card (`docs/MODEL_CARD.md`) y el informe de cierre (secciones 3.6 y 4.4). Responde cuatro preguntas que pide la rúbrica del P4 (evidencias A2.3 y A2.5 a A2.7):

1. ¿El orden sugerido encuentra más proveedores con una señal adversa documentada que la regla por monto, con el mismo trabajo? (línea de base)
2. ¿Con qué incertidumbre? (intervalo al 90 %)
3. ¿Dónde acierta y dónde no? (error por segmento)
4. ¿Por qué la evaluación no ve información posterior a la decisión? (sin fuga)

**Rótulo que no se quita:** 2022 y 2024 son una *segunda medición, no ciega*. El diseño de la lista ya conocía los exámenes de esos años (HALLAZGOS 160, secciones 54 y 78). La única prueba ciega es la lista de 2025, congelada sin etiquetas.

**Qué necesita.** Los datos no están en el repositorio porque traen RUC y nombres. Además del paquete del caso real (`docs/paquete_de_datos.md`), este notebook lee tres archivos de `datos/procesados/v4/resultados/lista_unica_limpia/`: `evaluacion_2022.parquet`, `evaluacion_2024.parquet` y `evaluacion_manifiesto.json`, que arma `radar/v4/tabla_evaluacion.py`, y `segunda_medicion.json`, para comparar. Si los datos están en otra carpeta, se indica con la variable de entorno `LLIKA_DATOS`.

**Qué imprime.** Solo conteos y razones: ningún RUC, nombre ni DNI.

In [1]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

raiz = Path.cwd()
while not (raiz / "llika.py").exists() and raiz != raiz.parent:
    raiz = raiz.parent
DATOS = Path(os.environ.get("LLIKA_DATOS", raiz / "datos"))
SAL = DATOS / "procesados" / "v4" / "resultados" / "lista_unica_limpia"
VERDADES = {"señal adversa": "y", "impedimento": "y_imp_real", "sanción": "y_S", "penalidad": "y_P", "observación de control": "y_rev"}
LISTAS = {"LLIKA": "en_lista", "monto": "en_monto", "tamaño": "en_tamano", "alerta y monto": "en_alerta_monto", "azar": "en_azar"}

def huella(f):
    h = hashlib.sha256()
    with open(f, "rb") as a:
        for b in iter(lambda: a.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()[:16]

man = json.loads((SAL / "evaluacion_manifiesto.json").read_text(encoding="utf-8"))
for nombre, info in man["archivos"].items():
    assert huella(SAL / nombre) == info["sha256"], f"{nombre} no tiene la huella del manifiesto"
print("Huellas comprobadas:", {n: i["sha256"] for n, i in man["archivos"].items()})
print("Al armar la tabla, la lista rehecha fue idéntica a la guardada:", {a: c["llaves_distintas"] + c["motivos_distintos"] + c["ordenes_distintas"] == 0 for a, c in man["comprobado"].items()})

Huellas comprobadas: {'evaluacion_2022.parquet': '96d6e4936f1e8dc9', 'evaluacion_2024.parquet': '0c52002e1bf62336'}
Al armar la tabla, la lista rehecha fue idéntica a la guardada: {'2022': True, '2024': True}


In [2]:
t = pd.concat([pd.read_parquet(SAL / f"evaluacion_{a}.parquet") for a in (2022, 2024)], ignore_index=True)
for c in list(VERDADES.values()) + list(LISTAS.values()):
    t[c] = t[c].astype(bool)
t["e_tipo"] = t["e_tipo"].astype(str).str.replace("\ufffd", "ó", regex=False)
resumen = t.groupby("anio").agg(casos=("grupo", "size"), entidades_anio=("grupo", "nunique"), filas_LLIKA=("en_lista", "sum"))
resumen

,casos,entidades_anio,filas_LLIKA
anio,,,
2022,769393,2518,12492
2024,1055732,2772,13811


## 1. Por qué la evaluación no ve el futuro

La decisión se toma al cierre del año, cuando el especialista arma el plan anual. Todo se corta en esa fecha:

| Lista | Años de ajuste de los modelos | Variables | Etiqueta del ajuste | Etiqueta de la prueba |
|---|---|---|---|---|
| 2022 | 2018 a 2020 (su ventana de dos años cerró al 31/12/2022) | Lo sabido al 31/12/2022 | Lo conocido al 31/12/2022 | Hechos de 2022 conocidos dentro de dos años |
| 2024 | 2018 a 2022 | Lo sabido al 31/12/2024 | Lo conocido al 31/12/2024 | Hechos de 2024 conocidos hasta el 31/10/2026 (22 meses) |
| 2025 | 2018 a 2023 | Lo sabido al 31/12/2025 | Lo conocido al 31/12/2025 | Ninguna: la lista se congeló sin etiquetas |

Además, el caso cuyo hecho ya era público al cierre no se aprende ni se evalúa (`radar/v4/evaluacion.py`, función `armar_corte`), y los antecedentes del proveedor entran solo si se publicaron hasta el 31/12 (Anexo D, Tabla D2, del informe). La partición es temporal, nunca al azar. Abajo se comprueba lo que se puede comprobar con estos archivos.

In [3]:
cuantos = t.groupby(["anio", "grupo"])[list(LISTAS.values())].sum()
print("Filas por entidad y año, como máximo:", cuantos.max().to_dict())
print("Filas de cada lista:", t[list(LISTAS.values())].sum().to_dict())
lista_2025 = pd.read_parquet(SAL / "lista_2025_unica.parquet")
etiquetas_2025 = [c for c in lista_2025.columns if c.startswith(("y", "fuentes", "conocido", "peso"))]
print("Columnas de etiqueta en la lista congelada de 2025:", etiquetas_2025, "| huella:", huella(SAL / "lista_2025_unica.parquet"))

Filas por entidad y año, como máximo: {'en_lista': 5, 'en_monto': 5, 'en_tamano': 5, 'en_alerta_monto': 5, 'en_azar': 5}
Filas de cada lista: {'en_lista': 26303, 'en_monto': 26303, 'en_tamano': 26303, 'en_alerta_monto': 26303, 'en_azar': 26303}
Columnas de etiqueta en la lista congelada de 2025: [] | huella: 189af867041d1270


## 2. La métrica y la línea de base

**Métrica técnica:** proveedores distintos con una señal adversa documentada entre los cinco primeros de cada entidad y año, contando a cada proveedor una vez por año. **Línea de base:** la regla por monto, que mira primero lo más caro, como hace hoy el especialista. Se comparan además el tamaño del proveedor en el Estado, la alerta seguida del monto y el azar, siempre con cinco por entidad.

**Incertidumbre:** las entidades y años son un censo; lo incierto es qué proveedores tienen señal. Por eso el intervalo al 90 % de cada razón sale de 2 000 remuestreos de Poisson por proveedor, con la semilla 0 (el mismo código de `radar/v4/lista_unica.py`, función `ic_proveedores`).

In [4]:
def proveedores(L, y="y"):
    return L.loc[L[y], ["ruc", "anio"]].drop_duplicates()

def razon(A, B, y="y", n=2000, semilla=0):
    a = A.loc[A[y]].groupby(["ruc", "anio"]).size().reset_index()["ruc"]
    b = B.loc[B[y]].groupby(["ruc", "anio"]).size().reset_index()["ruc"]
    if len(b) == 0:
        return np.nan, (np.nan, np.nan)
    f, u = pd.factorize(pd.concat([a, b]))
    c = len(a)
    rng = np.random.default_rng(semilla)
    bs = []
    for _ in range(n):
        w = rng.poisson(1.0, len(u))
        d0 = w[f[c:]].sum() / len(B)
        bs.append((w[f[:c]].sum() / len(A)) / d0 if d0 > 0 else np.nan)
    return (len(a) / len(A)) / (len(b) / len(B)), (np.nanpercentile(bs, 5), np.nanpercentile(bs, 95))

L = {n: t[t[col]] for n, col in LISTAS.items()}
tabla = pd.DataFrame({n: [len(proveedores(x, y)) for y in VERDADES.values()] for n, x in L.items()}, index=list(VERDADES))
for c in ("monto", "tamaño"):
    r = [razon(L["LLIKA"], L[c], y) for y in VERDADES.values()]
    tabla[f"LLIKA frente a {c} [IC 90 %]"] = [f"{x:.2f} [{lo:.2f} a {hi:.2f}]" if x == x else "" for x, (lo, hi) in r]
tabla

,LLIKA,monto,tamaño,alerta y monto,azar,LLIKA frente a monto [IC 90 %],LLIKA frente a tamaño [IC 90 %]
señal adversa,412,150,160,180,72,2.75 [2.43 a 3.14],2.58 [2.31 a 2.90]
impedimento,61,5,0,64,4,12.20 [6.67 a 31.97],
sanción,102,19,40,24,21,5.37 [3.92 a 8.23],2.55 [2.04 a 3.32]
penalidad,240,68,103,68,39,3.53 [2.98 a 4.33],2.33 [2.04 a 2.68]
observación de control,61,68,24,69,10,0.90 [0.75 a 1.05],2.54 [1.88 a 3.62]


In [5]:
guardada = json.loads((SAL / "segunda_medicion.json").read_text(encoding="utf-8"))
nombres = {"LLIKA": "lista única", "monto": "monto", "tamaño": "tamaño", "alerta y monto": "alerta y monto", "azar": "azar"}
claves = dict(zip(VERDADES, guardada["por_verdad"]))
for v in VERDADES:
    for n in LISTAS:
        assert tabla.loc[v, n] == guardada["por_verdad"][claves[v]][nombres[n]]["proveedores"], (v, n)
x, (lo, hi) = razon(L["LLIKA"], L["monto"])
g = guardada["razones"]["señal adversa"]["monto"]
assert abs(x - g["razon"]) < 1e-12 and abs(lo - g["ic90"][0]) < 1e-12 and abs(hi - g["ic90"][1]) < 1e-12
print("Repite la segunda medición guardada: mismos conteos y misma razón con su intervalo.")

Repite la segunda medición guardada: mismos conteos y misma razón con su intervalo.


## 3. Por año

In [6]:
filas = []
for a, g in t.groupby("anio"):
    La, Ma, Ta = g[g["en_lista"]], g[g["en_monto"]], g[g["en_tamano"]]
    x, (lo, hi) = razon(La, Ma)
    z, (zlo, zhi) = razon(La, Ta)
    filas.append({"año": a, "LLIKA": len(proveedores(La)), "monto": len(proveedores(Ma)), "tamaño": len(proveedores(Ta)), "azar": len(proveedores(g[g["en_azar"]])),
                  "frente al monto [IC 90 %]": f"{x:.2f} [{lo:.2f} a {hi:.2f}]", "frente al tamaño [IC 90 %]": f"{z:.2f} [{zlo:.2f} a {zhi:.2f}]"})
pd.DataFrame(filas).set_index("año")

,LLIKA,monto,tamaño,azar,frente al monto [IC 90 %],frente al tamaño [IC 90 %]
año,,,,,,
2022,208,95,92,45,2.19 [1.88 a 2.57],2.26 [1.94 a 2.64]
2024,204,55,68,27,3.71 [3.05 a 4.67],3.00 [2.52 a 3.65]


## 4. Indicador de impacto y métrica de contrapeso

**Indicador de impacto:** revisiones en vano por cada acierto. Cada proveedor revisado sin señal es una carpeta en vano; a 40 o 60 minutos por proveedor (lo que dijeron los especialistas de la prueba de uso), se traduce en horas.

**Contrapeso:** las observaciones de control, que son la verdad propia de la Contraloría, no deben empeorar frente al monto. También se muestran los aciertos por fila (cuenta las repeticiones de un mismo proveedor, donde el tamaño gana) y cuántos proveedores con señal no ve la lista.

In [7]:
filas_lista = int(t["en_lista"].sum())
imp = []
for n, x in L.items():
    ac = len(proveedores(x))
    vano = (len(x) - ac) / ac
    imp.append({"lista": n, "filas": len(x), "aciertos": ac, "en vano por acierto": round(vano, 1),
                "horas por acierto (40 a 60 min)": f"{vano * 40 / 60:.0f} a {vano:.0f}", "aciertos por fila": int(x["y"].sum())})
display(pd.DataFrame(imp).set_index("lista"))
x, (lo, hi) = razon(L["LLIKA"], L["monto"], "y_rev")
print(f"Observaciones de control, LLIKA frente al monto: {x:.2f} [{lo:.2f} a {hi:.2f}]")

,filas,aciertos,en vano por acierto,horas por acierto (40 a 60 min),aciertos por fila
lista,,,,,
LLIKA,26303,412,62.8,42 a 63,657
monto,26303,150,174.4,116 a 174,222
tamaño,26303,160,163.4,109 a 163,1033
alerta y monto,26303,180,145.1,97 a 145,249
azar,26303,72,364.3,243 a 364,85


Observaciones de control, LLIKA frente al monto: 0.90 [0.75 a 1.05]


## 5. Error por segmento

Para cada segmento: filas de la lista, proveedores con señal y aciertos por cada 100 filas; los proveedores con señal de la regla por monto y del tamaño, con el mismo número de filas, y la razón de las precisiones con su intervalo. Un segmento con pocos aciertos tiene un intervalo ancho: se lee con eso a la vista.

In [8]:
def por_segmento(columna, orden=None):
    filas = []
    for s, g in t.groupby(columna, observed=True):
        La, Ma, Ta = g[g["en_lista"]], g[g["en_monto"]], g[g["en_tamano"]]
        if len(La) == 0 or len(Ma) == 0:
            continue
        a, m, z = len(proveedores(La)), len(proveedores(Ma)), len(proveedores(Ta))
        x, (lo, hi) = razon(La, Ma)
        w, (wlo, whi) = razon(La, Ta)
        filas.append({columna: s, "filas": len(La), "LLIKA": a, "por 100 filas": round(100 * a / len(La), 2), "monto": m, "tamaño": z,
                      "frente al monto [IC 90 %]": f"{x:.2f} [{lo:.2f} a {hi:.2f}]" if x == x else "sin aciertos del monto",
                      "frente al tamaño [IC 90 %]": f"{w:.2f} [{wlo:.2f} a {whi:.2f}]" if w == w else "sin aciertos del tamaño"})
    r = pd.DataFrame(filas).set_index(columna)
    return r.loc[orden] if orden else r

casos = t.groupby(["anio", "grupo"])["grupo"].transform("size")
t["tamaño de la entidad"] = pd.cut(casos, [0, 100, 500, np.inf], labels=["hasta 100 proveedores", "101 a 500", "más de 500"])
por_segmento("tamaño de la entidad", ["hasta 100 proveedores", "101 a 500", "más de 500"])

,filas,LLIKA,por 100 filas,monto,tamaño,frente al monto [IC 90 %],frente al tamaño [IC 90 %]
tamaño de la entidad,,,,,,,
hasta 100 proveedores,7613,77,1.01,38,62,2.03 [1.65 a 2.56],1.24 [1.03 a 1.51]
101 a 500,14275,267,1.87,77,125,3.47 [2.94 a 4.15],2.14 [1.88 a 2.47]
más de 500,4415,164,3.71,58,56,2.83 [2.31 a 3.57],2.93 [2.37 a 3.71]


In [9]:
por_segmento("e_tipo").sort_values("filas", ascending=False)

,filas,LLIKA,por 100 filas,monto,tamaño,frente al monto [IC 90 %],frente al tamaño [IC 90 %]
e_tipo,,,,,,,
municipalidad distrital,13538,119,0.88,42,51,2.83 [2.25 a 3.64],2.33 [1.87 a 2.92]
gobierno regional,5369,99,1.84,38,66,2.61 [2.10 a 3.44],1.50 [1.24 a 1.85]
otra,3772,170,4.51,55,77,3.09 [2.55 a 3.91],2.21 [1.87 a 2.66]
municipalidad provincial,1860,32,1.72,12,27,2.67 [1.87 a 4.33],1.19 [0.86 a 1.64]
universidad,550,18,3.27,3,14,6.00 [3.09 a 18.00],1.29 [0.77 a 2.22]
programa o proyecto,459,24,5.23,4,17,6.00 [3.43 a 18.00],1.41 [1.00 a 2.22]
salud,370,23,6.22,11,15,2.09 [1.40 a 3.57],1.53 [0.95 a 2.60]
ministerio,315,25,7.94,6,17,4.17 [2.50 a 10.00],1.47 [0.88 a 2.62]
educación,70,5,7.14,3,4,1.67 [1.00 a 4.00],1.25 [0.40 a 4.00]


In [10]:
t["tipo de proveedor"] = np.where(t["u_persona_natural"].astype(bool), "persona natural", "empresa u otro")
por_segmento("tipo de proveedor")

,filas,LLIKA,por 100 filas,monto,tamaño,frente al monto [IC 90 %],frente al tamaño [IC 90 %]
tipo de proveedor,,,,,,,
empresa u otro,18640,338,1.81,123,148,2.35 [2.05 a 2.72],2.86 [2.55 a 3.24]
persona natural,7663,74,0.97,27,12,3.71 [2.80 a 5.27],2.39 [1.64 a 4.10]


**Por motivo del puesto.** El motivo solo existe en la lista: dice por qué cada proveedor quedó entre los cinco primeros. «Historial» se parte según el proveedor tenga o no antecedentes en la ficha.

In [11]:
H = ["h_prov_penalidades", "h_prov_sanciones_otras", "h_prov_multas", "h_prov_inhabilitaciones"]
fl = pd.concat([pd.read_parquet(SAL / f"lista_{a}.parquet", columns=["ruc_entidad", "ruc", "anio"] + H) for a in (2022, 2024)])
Lm = L["LLIKA"].merge(fl.astype({"ruc_entidad": str, "ruc": str, "anio": int}), on=["ruc_entidad", "ruc", "anio"], how="left", validate="one_to_one")
assert len(Lm) == len(L["LLIKA"]) and Lm["h_prov_penalidades"].notna().all()
con = Lm[H].fillna(0).sum(axis=1) > 0
hist = Lm["motivo"].eq("historial del proveedor")
Lm.loc[hist, "motivo"] = np.where(con[hist], "historial, con antecedentes", "historial, sin antecedentes")
mot = []
for m, g in Lm.groupby("motivo"):
    a = len(proveedores(g))
    mot.append({"motivo": m, "filas": len(g), "proveedores con señal": a, "por 100 filas": round(100 * a / len(g), 1)})
pd.DataFrame(mot).set_index("motivo").sort_values("por 100 filas", ascending=False)

,filas,proveedores con señal,por 100 filas
motivo,,,
"historial, con antecedentes",3586,158,4.4
vínculo: autoridad o su empresa,196,8,4.1
"vínculo: pariente, es una coincidencia",1161,25,2.2
"historial, sin antecedentes",10933,179,1.6
compra con bandera,4394,25,0.6
sin señal fuerte: peso en la entidad,5198,20,0.4
sin señal fuerte: compra más grande,835,3,0.4


**Lo que la lista no ve.** De todos los proveedores con señal en los casos evaluados, cuántos quedan entre los cinco primeros de LLIKA, por tipo de señal. El techo es lo máximo que vería una lista perfecta con cinco por entidad (flujo máximo entre proveedores con señal y entidades con cinco puestos).

In [12]:
def techo(q, y="y", k=5):
    e = q[q[y]][["ruc", "anio", "grupo"]].drop_duplicates()
    pi, pu = pd.factorize(e["ruc"] + "|" + e["anio"].astype(str))
    gi, gu = pd.factorize(e["grupo"])
    n_p, n_g = len(pu), len(gu)
    fin = 1 + n_p + n_g
    filas = np.r_[np.zeros(n_p, dtype=int), 1 + pi, 1 + n_p + np.arange(n_g)]
    cols = np.r_[1 + np.arange(n_p), 1 + n_p + gi, np.full(n_g, fin)]
    cap = np.r_[np.ones(n_p, dtype=int), np.ones(len(e), dtype=int), np.full(n_g, k)]
    return int(maximum_flow(csr_matrix((cap.astype(np.int32), (filas, cols)), shape=(fin + 1, fin + 1)), 0, fin).flow_value)

nv = []
for v, y in VERDADES.items():
    u, l = len(proveedores(t, y)), len(proveedores(L["LLIKA"], y))
    nv.append({"señal": v, "con señal en los casos": u, "LLIKA ve": l, "no ve": u - l, "alcance (%)": round(100 * l / u, 1)})
nv = pd.DataFrame(nv).set_index("señal")
nv["techo con 5 por entidad"] = [str(techo(t, y)) if v == "señal adversa" else "" for v, y in VERDADES.items()]
nv

,con señal en los casos,LLIKA ve,no ve,alcance (%),techo con 5 por entidad
señal,,,,,
señal adversa,1254,412,842,32.9,1153
impedimento,234,61,173,26.1,
sanción,213,102,111,47.9,
penalidad,412,240,172,58.3,
observación de control,588,61,527,10.4,


## 6. Lectura

Escrita después de ver las cifras de arriba; si los datos cambian, hay que volver a escribirla.

1. **Frente a la línea de base.** Con el mismo trabajo, cinco proveedores por entidad y año, LLIKA halla 412 proveedores con una señal adversa documentada y la regla por monto, 150: 2,75 veces, con un intervalo al 90 % de 2,43 a 3,14. Gana en los dos años; en 2024 la ventaja es mayor sobre todo porque el monto acierta menos.
2. **Impacto.** LLIKA revisa 62,8 carpetas en vano por cada acierto y el monto, 174,4: de 42 a 63 horas por acierto frente a 116 a 174.
3. **Contrapeso.** En las observaciones de control casi empata con el monto (0,90, de 0,75 a 1,05): no las empeora de forma clara, pero tampoco las mejora. Contando filas y no proveedores, el tamaño sigue arriba (1 033 frente a 657), porque repite a los mismos proveedores grandes en muchas entidades.
4. **Por segmento.** Le gana al monto en todos los tamaños y tipos de entidad, con personas naturales y con empresas. Frente al tamaño la ventaja es clara en las entidades con más de 100 proveedores, en las municipalidades distritales y en los gobiernos regionales; en las entidades con hasta 100 proveedores se achica (1,24, de 1,03 a 1,51), y en municipalidades provinciales, universidades, salud, ministerios y educación el intervalo incluye el 1: ahí no se puede decir que le gane al tamaño. Las filas de personas naturales aciertan la mitad que las de empresas (0,97 frente a 1,81 por cada 100 filas).
5. **Por motivo.** El historial con antecedentes y el vínculo con una autoridad rinden más de 4 aciertos por cada 100 filas; la compra con bandera y las filas sin señal fuerte, que son el 40 % de la lista, rinden 0,4 a 0,6, cerca del azar (0,27).
6. **Lo que no ve.** Deja pasar 842 de los 1 254 proveedores con señal (67 %), y en las observaciones de control ve 1 de cada 10. Una lista perfecta con cinco por entidad vería 1 153: el límite no es el cupo de cinco, es lo que las señales alcanzan a ver.
7. **No es una prueba ciega.** La prueba ciega es la lista de 2025, congelada sin etiquetas (huella `189af867041d1270`).
